<a href="https://www.kaggle.com/code/vedikagupta0/curated-baseline-model-logreg-0-76?scriptVersionId=304333869" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [69]:
import numpy as np
import pandas as pd
df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
tf = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

In [70]:
df.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [71]:
id_col = ['id']

In [72]:
df.describe(include='object')

,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Churn
count,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194
unique,2,2,2,2,3,3,3,3,3,3,3,3,3,2,4,2
top,Female,Yes,No,Yes,No,Fiber optic,No,No,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,No
freq,298738,309554,414362,557893,283384,272386,289474,250083,247377,288571,240301,241435,298918,365579,215372,460377


In [73]:
cat_cols = df.describe(include='object').columns[:-1]

In [74]:
from sklearn.model_selection import train_test_split
Xtr, Xte, ytr, yte = train_test_split(df.drop(columns=['Churn', 'id']), df['Churn'], test_size=0.2, random_state=42)

In [75]:
ytr.value_counts()

Churn
No     368442
Yes    106913
Name: count, dtype: int64

In [76]:
yte.value_counts()

Churn
No     91935
Yes    26904
Name: count, dtype: int64

##### major imbalance is observed here

In [77]:
from sklearn.preprocessing import LabelEncoder

for col in cat_cols:
    le = LabelEncoder()
    Xtr[col] = le.fit_transform(Xtr[col])
    Xte[col] = le.transform(Xte[col])
    tf[col] = le.transform(tf[col])

In [78]:
from sklearn.linear_model import LogisticRegression
logreg = LogisticRegression()
logreg.fit(Xtr, ytr)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [79]:
y_pred = logreg.predict(Xte)

In [80]:
from sklearn.metrics import classification_report
print(classification_report(yte, y_pred))

              precision    recall  f1-score   support

          No       0.90      0.91      0.90     91935
         Yes       0.67      0.64      0.65     26904

    accuracy                           0.85    118839
   macro avg       0.78      0.77      0.78    118839
weighted avg       0.85      0.85      0.85    118839



In [81]:
baseline_submission = pd.DataFrame({
    'id': tf.pop('id'),
    'Churn': pd.Series(logreg.predict(tf)).map({'No':0, 'Yes':1})
})

In [82]:
baseline_submission.head()

,id,Churn
0,594194,0
1,594195,0
2,594196,0
3,594197,0
4,594198,0


In [83]:
baseline_submission.to_csv('baseline.csv', index=False)